In [ ]:
import hashlib
import os
import re
import time
from typing import Any

from dotenv import load_dotenv
from genai_core.memory import EpisodicMemory
from genai_core.memory.in_memory_store import InMemoryMemoryStore
from google.genai import Client, types
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

True

In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)

### Task 1

In [3]:
class Critique(BaseModel):
    """Pydantic model for the critique request."""

    passed: bool = Field(..., description="Indicates if the code passed the test cases.")
    issues: list[str] = Field(default_factory=list, description="List of issues found in the code.")
    severity: int = Field(ge=0, le=3, description="Severity level of the issues found (0-3).")


def reflection(
    goal: str,
    llm: Client,
    critique: type[Critique] = Critique,
    max_round: int = 3,
    model: str = "gemini-3.5-flash-lite",
) -> str:
    """Generate -> Critique -> Revise loop with a round cap and Pydantic Critique model."""
    critique_cls = critique if isinstance(critique, type) else type(critique)

    # 1. Generate initial draft
    print("--- Round 1: Generate ---")
    gen_response = llm.models.generate_content(
        model=model,
        contents=f"Goal: {goal}\n\nProvide the best complete solution.",
    )
    current_solution = gen_response.text or ""
    print(f"Initial Draft:\n{current_solution}\n")

    for round_num in range(1, max_round + 1):
        # 2. Critique current draft
        critique_prompt = (
            f"Goal: {goal}\n\n"
            f"Current Solution:\n{current_solution}\n\n"
            "Evaluate whether this meets all requirements and identify any bugs or missing edge cases.\n"
            "Set passed=True only if the solution is completely correct and has no issues."
        )
        critique_response = llm.models.generate_content(
            model=model,
            contents=critique_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=critique_cls,
            ),
        )
        critique_result = getattr(
            critique_response, "parsed", None
        ) or critique_cls.model_validate_json(critique_response.text)

        print(f"--- Round {round_num} Critique ---")
        print(
            f"Passed: {critique_result.passed} | Severity: {critique_result.severity} | Issues: {critique_result.issues}\n"
        )

        # Check if passed or round cap reached
        if critique_result.passed or round_num == max_round:
            if critique_result.passed:
                print(f"Critique passed in round {round_num}!")
            else:
                print(f"Reached round cap ({max_round}). Returning latest revision.")
            break

        # 3. Revise based on critique feedback
        revise_prompt = (
            f"Goal: {goal}\n\n"
            f"Current Solution:\n{current_solution}\n\n"
            f"Critique Feedback:\n"
            f"- Issues: {critique_result.issues}\n"
            f"- Severity: {critique_result.severity}\n\n"
            "Revise and improve the solution addressing every issue listed above."
        )
        revised_response = llm.models.generate_content(
            model=model,
            contents=revise_prompt,
        )
        current_solution = revised_response.text or ""
        print(f"--- Round {round_num + 1}: Revise ---")
        print(f"Revised Draft:\n{current_solution}\n")

    return current_solution

In [4]:
# Example task: generate, critique, and revise an email domain extractor
task_prompt = (
    "Write a Python function `extract_domain(email: str) -> str | None` "
    "that extracts the clean domain name from an email address string. "
    "Handle edge cases such as leading/trailing whitespace, missing '@', multiple '@' symbols, "
    "subdomains, invalid domain endings (e.g. trailing dots), and return None for any invalid email."
)

result = reflection(goal=task_prompt, llm=google_client, max_round=3)
print("=== Final Accepted Solution ===")
print(result)

--- Round 1: Generate ---
Initial Draft:
Here is a robust, production-ready solution to extract the clean domain name from an email address in Python. 

This solution uses Python's built-in `email.utils` library combined with strict validation checks to handle standard edge cases (multiple `@` symbols, whitespace, trailing dots, subdomains, and invalid structures).

### Complete Solution

```python
from email.utils import parseaddr


def extract_domain(email: str) -> str | None:
    """Extracts the clean domain name from an email address string.

    Returns None for invalid emails.
    """
    if not isinstance(email, str):
        return None

    # Strip leading/trailing whitespace
    email = email.strip()

    if not email:
        return None

    # Handle display names (e.g., "John Doe <john@example.com>") using email.utils
    _, parsed_email = parseaddr(email)

    # Ensure there is exactly one '@' in the parsed address
    if parsed_email.count("@") != 1:
        return None


### Task 2

In [5]:
class UserProfile(BaseModel):
    """Schema for user profile."""

    username: str = Field(..., min_length=3, description="Username (at least 3 characters)")
    email: str = Field(..., description="User email address")
    age: int = Field(..., ge=1, le=120, description="User age in years")
    role: str = Field(..., description="Role: admin, editor, or viewer")


def deterministic_verify(raw_output: str) -> tuple[bool, list[str], UserProfile | None]:
    """Deterministic verifier: schema validation plus one business rule."""
    errors: list[str] = []

    # Clean markdown fences
    text = re.sub(r"^```json\s*", "", raw_output.strip(), flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text).strip()

    # 1. Schema validation (Pydantic)
    try:
        profile = UserProfile.model_validate_json(text)
    except ValidationError as e:
        for err in e.errors():
            loc = " -> ".join(str(line) for line in err["loc"])
            errors.append(f"Schema Error [{loc}]: {err['msg']}")
        return False, errors, None
    except Exception as e:  # noqa: BLE001
        return False, [f"JSON Syntax Error: {e!s}"], None

    # 2. Business rule: admin role requires age >= 21
    if profile.role.lower() == "admin" and profile.age < 21:
        errors.append(
            f"Business Rule Violation: 'admin' role requires age >= 21 (current age: {profile.age})."
        )

    if errors:
        return False, errors, profile
    return True, [], profile


def reflection_with_verifier(
    goal: str,
    llm: Client,
    max_round: int = 3,
    model: str = "gemini-3.5-flash-lite",
) -> str:
    """Generate -> Deterministic Verify -> Revise loop passing exact errors into revision prompt."""
    # 1. Generate initial solution
    print("--- Round 1: Generate ---")
    gen_response = llm.models.generate_content(
        model=model,
        contents=f"Goal: {goal}\n\nReturn ONLY a valid JSON object matching the UserProfile schema.",
    )
    current = gen_response.text or ""
    print(f"Initial Draft:\n{current}\n")

    for round_num in range(1, max_round + 1):
        # 2. Deterministic Verification
        passed, errors, _ = deterministic_verify(current)
        print(f"--- Round {round_num} Deterministic Verification ---")
        print(f"Passed: {passed} | Errors: {errors}\n")

        if passed or round_num == max_round:
            if passed:
                print(f"Verification passed in round {round_num}!")
            else:
                print(f"Reached round cap ({max_round}). Returning latest output.")
            break

        # 3. Revise with exact verifier errors in the prompt
        error_list_str = "\n".join(f"- {err}" for err in errors)
        revise_prompt = (
            f"Goal: {goal}\n\n"
            f"Previous Output:\n{current}\n\n"
            f"Deterministic Verifier Errors to Fix:\n{error_list_str}\n\n"
            "Fix all schema and business rule errors and return ONLY the corrected JSON object."
        )
        rev_response = llm.models.generate_content(
            model=model,
            contents=revise_prompt,
        )
        current = rev_response.text or ""
        print(f"--- Round {round_num + 1}: Revise ---")
        print(f"Revised Draft:\n{current}\n")

    return current

In [6]:
# Example task: triggers business rule check (admin with age 19)
task_goal = (
    "Generate a UserProfile JSON for a user named 'alex_dev', email 'alex@company.com', "
    "age 19, who requested the 'admin' role."
)

verified_output = reflection_with_verifier(
    goal=task_goal,
    llm=google_client,
    max_round=3,
)

print("=== Final Verified Output ===")
print(verified_output)

--- Round 1: Generate ---
Initial Draft:
{
  "username": "alex_dev",
  "email": "alex@company.com",
  "age": 19,
  "requested_role": "admin"
}

--- Round 1 Deterministic Verification ---
Passed: False | Errors: ['Schema Error [role]: Field required']

--- Round 2: Revise ---
Revised Draft:
{
  "username": "alex_dev",
  "email": "alex@company.com",
  "age": 19,
  "requested_role": "admin",
  "role": "user"
}

--- Round 2 Deterministic Verification ---
Passed: True | Errors: []

Verification passed in round 2!
=== Final Verified Output ===
{
  "username": "alex_dev",
  "email": "alex@company.com",
  "age": 19,
  "requested_role": "admin",
  "role": "user"
}


### Task 3

In [ ]:
# Cost constants for gemini-3.5-flash-lite ($ per 1M tokens)
INPUT_COST_PER_M = 0.075
OUTPUT_COST_PER_M = 0.30


def run_single_evaluation(
    goal: str,
    llm: Client,
    model: str = "gemini-3.5-flash-lite",
    max_retries: int = 3,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """Run a single input both without reflection and with reflection, tracking validity, latency, and cost."""

    def call_api(prompt: str) -> tuple[str, float, int, int]:
        for attempt in range(max_retries):
            try:
                start_t = time.perf_counter()
                resp = llm.models.generate_content(model=model, contents=prompt)
                elapsed = time.perf_counter() - start_t
                in_tok = resp.usage_metadata.prompt_token_count if resp.usage_metadata else 0
                out_tok = resp.usage_metadata.candidates_token_count if resp.usage_metadata else 0
                return (resp.text or ""), elapsed, in_tok, out_tok
            except Exception as e:
                if (
                    "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)
                ) and attempt < max_retries - 1:
                    time.sleep(15 * (attempt + 1))
                else:
                    raise
        raise RuntimeError("Rate limit retries exceeded")

    # 1. Initial generation (Zero-shot / Without Reflection)
    gen_prompt = (
        f"Goal: {goal}\n\n"
        "Return ONLY a valid JSON object matching the UserProfile schema:\n"
        "- username: str (min 3 chars)\n"
        "- email: str\n"
        "- age: int (1-120)\n"
        "- role: 'admin', 'editor', or 'viewer'\n"
        "Note: Business rule requires that 'admin' role requires age >= 21."
    )
    init_text, init_lat, init_in, init_out = call_api(gen_prompt)
    is_valid_init, init_errs, _ = deterministic_verify(init_text)
    init_cost = (init_in / 1_000_000 * INPUT_COST_PER_M) + (
        init_out / 1_000_000 * OUTPUT_COST_PER_M
    )

    no_reflect = {
        "valid": is_valid_init,
        "latency": init_lat,
        "cost": init_cost,
        "tokens": init_in + init_out,
    }

    # 2. With Reflection (Revision loop if deterministic verification fails)
    ref_valid = is_valid_init
    ref_lat = init_lat
    ref_cost = init_cost
    ref_tokens = init_in + init_out

    if not is_valid_init:
        time.sleep(4.2)  # Pace between requests for free-tier quota
        rev_prompt = (
            f"Goal: {goal}\n\n"
            f"Your previous output was:\n{init_text}\n\n"
            "It failed deterministic verification with these exact errors:\n"
            + "\n".join(f"- {e}" for e in init_errs)
            + "\n\nPlease revise the JSON so all schema constraints and business rules are strictly satisfied.\n"
            "Return ONLY the revised valid JSON object."
        )
        rev_text, rev_lat, r_in, r_out = call_api(rev_prompt)
        ref_lat += rev_lat
        ref_tokens += r_in + r_out
        ref_cost += (r_in / 1_000_000 * INPUT_COST_PER_M) + (r_out / 1_000_000 * OUTPUT_COST_PER_M)
        is_valid_rev, _, _ = deterministic_verify(rev_text)
        ref_valid = is_valid_rev

    reflect = {
        "valid": ref_valid,
        "latency": ref_lat,
        "cost": ref_cost,
        "tokens": ref_tokens,
    }

    return no_reflect, reflect

In [7]:
benchmark_inputs = [
    "Create an admin profile for Sarah, age 19, email sarah@tech.com",
    "Create an editor profile for Ed, age 30, email ed@press.co",
    "Create a profile for viewer Alex, age 25, email alex@view.org",
    "Create an admin profile for junior apprentice Kyle who is 18 years old, email kyle@corp.io",
    "Create a viewer profile for baby Emma, age 0, email emma@family.net",
    "Create a profile for user Jo with role admin, age 28, email jo@admin.net",
    "Create an editor profile for Bob, age 45, email bob@edit.com",
    "Create an admin profile for 20-year-old college lead Sam, email sam@uni.edu",
    "Create a viewer profile for Grandpa Joe, age 125, email joe@senior.org",
    "Create a profile for Al, age 32, role editor, email al@news.org",
    "Create an admin profile for team captain Liam, age 22, email liam@firm.com",
    "Create an admin profile for student intern Maya, age 17, email maya@school.org",
    "Create a viewer profile for Ty, age 24, email ty@stream.tv",
    "Create an admin profile for prodigy coder Dan, age 16, email dan@ai.lab",
    "Create an editor profile for Clara, age 29, email clara@mag.com",
    "Create an admin profile for newcomer Leo, age 20, email leo@start.up",
    "Create a viewer profile for Xi, age 33, email xi@portal.cn",
    "Create an admin profile for trainee Zoe, age 19, email zoe@cloud.io",
    "Create an editor profile for Marcus, age 41, email marcus@media.co",
    "Create an admin profile for young researcher Ava, age 18, email ava@lab.org",
]

no_reflect_results = []
reflect_results = []

print(
    f"Benchmarking {len(benchmark_inputs)} inputs across Both Conditions (Without vs With Reflection)...\n"
)

for idx, prompt_text in enumerate(benchmark_inputs, 1):
    no_ref, ref = run_single_evaluation(prompt_text, google_client)
    no_reflect_results.append(no_ref)
    reflect_results.append(ref)
    status_no = "VALID" if no_ref["valid"] else "INVALID"
    status_ref = "VALID" if ref["valid"] else "INVALID"
    print(
        f"[{idx:02d}/20] No-Reflect: {status_no} ({no_ref['latency']:.2f}s) | With-Reflect: {status_ref} ({ref['latency']:.2f}s)"
    )
    if idx < len(benchmark_inputs):
        time.sleep(4.2)

# Compute comparative aggregate metrics
n = len(benchmark_inputs)
valid_no = sum(1 for r in no_reflect_results if r["valid"])
valid_ref = sum(1 for r in reflect_results if r["valid"])
avg_lat_no = sum(r["latency"] for r in no_reflect_results) / n
avg_lat_ref = sum(r["latency"] for r in reflect_results) / n
avg_cost_no = sum(r["cost"] for r in no_reflect_results) / n
avg_cost_ref = sum(r["cost"] for r in reflect_results) / n
tot_cost_no = sum(r["cost"] for r in no_reflect_results)
tot_cost_ref = sum(r["cost"] for r in reflect_results)

print("\n" + "=" * 65)
print("=== TASK 3 BENCHMARK COMPARISON ON 20 INPUTS ===")
print("=" * 65)
print(f"{'Metric':<25} | {'Without Reflection':<18} | {'With Reflection':<18}")
print("-" * 65)
print(
    f"{'Validity Rate':<25} | {valid_no}/{n} ({valid_no / n * 100:.1f}%)"
    + " " * 8
    + f"| {valid_ref}/{n} ({valid_ref / n * 100:.1f}%)"
)
print(f"{'Average Latency':<25} | {avg_lat_no:.3f} s" + " " * 11 + f"| {avg_lat_ref:.3f} s")
print(f"{'Average Cost / Req':<25} | ${avg_cost_no:.6f}" + " " * 7 + f"| ${avg_cost_ref:.6f}")
print(f"{'Total Cost (20 inputs)':<25} | ${tot_cost_no:.6f}" + " " * 7 + f"| ${tot_cost_ref:.6f}")
print("=" * 65)

Benchmarking 20 inputs across Both Conditions (Without vs With Reflection)...

[01/20] No-Reflect: VALID (1.19s) | With-Reflect: VALID (1.19s)
[02/20] No-Reflect: INVALID (0.71s) | With-Reflect: VALID (1.93s)
[03/20] No-Reflect: VALID (0.82s) | With-Reflect: VALID (0.82s)
[04/20] No-Reflect: VALID (1.94s) | With-Reflect: VALID (1.94s)
[05/20] No-Reflect: INVALID (1.45s) | With-Reflect: VALID (19.56s)
[06/20] No-Reflect: INVALID (1.53s) | With-Reflect: VALID (2.45s)
[07/20] No-Reflect: VALID (1.34s) | With-Reflect: VALID (1.34s)
[08/20] No-Reflect: VALID (0.91s) | With-Reflect: VALID (0.91s)
[09/20] No-Reflect: VALID (1.15s) | With-Reflect: VALID (1.15s)
[10/20] No-Reflect: INVALID (1.23s) | With-Reflect: VALID (2.27s)
[11/20] No-Reflect: VALID (1.71s) | With-Reflect: VALID (1.71s)
[12/20] No-Reflect: VALID (0.91s) | With-Reflect: VALID (0.91s)
[13/20] No-Reflect: VALID (0.93s) | With-Reflect: VALID (0.93s)
[14/20] No-Reflect: VALID (1.12s) | With-Reflect: VALID (1.12s)
[15/20] No-Refle

### Task 4

In [7]:
class ScoredCritique(BaseModel):
    """Critique model evaluating solution with a numeric score."""

    score: float = Field(ge=0.0, le=10.0, description="Quality score from 0.0 to 10.0")
    passed: bool = Field(
        ..., description="Whether the solution is completely acceptable (score >= 9.0)"
    )
    feedback: str = Field(..., description="Actionable critique feedback or suggested improvements")


def reflection_with_best_so_far(
    goal: str,
    llm: Client,
    max_round: int = 4,
    score_threshold: float = 9.0,
    model: str = "gemini-3.5-flash-lite",
) -> tuple[str, float]:
    """Reflection loop with oscillation detection and keep-the-best-so-far policy."""
    history_hashes: set[str] = set()
    best_solution: str = ""
    best_score: float = -1.0

    def get_hash(text: str) -> str:
        # Normalize whitespace for reliable oscillation detection
        normalized = " ".join(text.strip().split())
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    # 1. Generate initial solution
    print("--- Round 1: Generate ---")
    gen_res = llm.models.generate_content(
        model=model,
        contents=f"Goal: {goal}\n\nProvide the best complete solution.",
    )
    current = gen_res.text or ""
    print(f"Draft 1:\n{current}\n")

    for round_num in range(1, max_round + 1):
        # Oscillation Detection: check if current solution was already seen
        current_hash = get_hash(current)
        if current_hash in history_hashes:
            print(
                f"⚠️ Oscillation Detected in round {round_num}! Solution is repeating a prior state."
            )
            print("Terminating loop early to avoid cycling.\n")
            break
        history_hashes.add(current_hash)

        # 2. Critique and Numeric Scoring
        critique_prompt = (
            f"Goal: {goal}\n\n"
            f"Candidate Solution:\n{current}\n\n"
            "Score this solution strictly from 0.0 to 10.0 based on correctness, edge cases, and code clarity. "
            "Set passed=True only if score >= 9.0."
        )
        crit_res = llm.models.generate_content(
            model=model,
            contents=critique_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ScoredCritique,
            ),
        )
        evaluation = getattr(crit_res, "parsed", None) or ScoredCritique.model_validate_json(
            crit_res.text
        )

        print(f"--- Round {round_num} Evaluation ---")
        print(f"Score: {evaluation.score}/10.0 | Passed: {evaluation.passed}")
        print(f"Feedback: {evaluation.feedback}\n")

        # Keep-the-best-so-far policy: retain candidate with highest numeric score
        if evaluation.score > best_score:
            best_score = evaluation.score
            best_solution = current
            print(f"⭐ Updated best-so-far solution (New Best Score: {best_score}/10.0)\n")

        # Exit if target score reached or max rounds cap reached
        if evaluation.passed or evaluation.score >= score_threshold:
            print(f"Quality threshold reached in round {round_num}!")
            break
        if round_num == max_round:
            print(f"Reached round cap ({max_round}).")
            break

        # 3. Revise based on critique
        revise_prompt = (
            f"Goal: {goal}\n\n"
            f"Current Draft (Score {evaluation.score}/10.0):\n{current}\n\n"
            f"Critique Feedback:\n{evaluation.feedback}\n\n"
            "Revise and improve the solution to address all critique points."
        )
        rev_res = llm.models.generate_content(
            model=model,
            contents=revise_prompt,
        )
        current = rev_res.text or ""
        print(f"--- Round {round_num + 1}: Revise ---")
        print(f"Draft {round_num + 1}:\n{current}\n")

    return best_solution, best_score

In [8]:
# Task demonstrating numeric scoring, keep-the-best-so-far policy, and reflection
task = (
    "Write a concise Python function `parse_query_params(url: str) -> dict[str, list[str]]` "
    "that parses query parameters into a dictionary mapping keys to lists of values, "
    "handling URL decoding, parameters without values, empty queries, and fragment stripping."
)

best_code, best_score = reflection_with_best_so_far(
    goal=task,
    llm=google_client,
    max_round=3,
)

print(f"=== Final Best-So-Far Solution (Score: {best_score}/10.0) ===")
print(best_code)

--- Round 1: Generate ---
Draft 1:
Here is the best and most concise complete solution using Python's standard library `urllib.parse` module. 

It correctly strips URL fragments, parses percent-encoded keys and values, groups multiple values for the same key into a list, and safely handles edge cases like empty queries or parameters without values.

```python
from urllib.parse import parse_qs, urlparse


def parse_query_params(url: str) -> dict[str, list[str]]:
    """Parses query parameters from a URL into a dictionary mapping keys to lists of values.

    Handles URL decoding, parameters without values, empty queries, and fragment stripping.
    """
    return parse_qs(urlparse(url).query, keep_blank_values=True)
```

### Explanation:
1. **`urlparse(url)`**: Splits the URL into its components (scheme, netloc, path, params, query, fragment). It automatically ignores/strips the fragment.
2. **`.query`**: Extracts just the query string portion from the parsed result.
3. **`parse_qs(...,

### Task 5

In [9]:
class IssueDetail(BaseModel):
    """Specific issue requiring exact quoted evidence from the draft."""

    quoted_evidence: str = Field(
        ...,
        description="Exact verbatim substring quote from the draft demonstrating the flaw",
    )
    explanation: str = Field(
        ...,
        description="Explanation of why this quoted snippet is an issue or bug",
    )


class EvidenceCriticResult(BaseModel):
    """Critic evaluation permitting 'no issues' and requiring quoted evidence."""

    has_issues: bool = Field(
        ...,
        description="False if the draft has NO issues, True if issues exist",
    )
    issues: list[IssueDetail] = Field(
        default_factory=list,
        description="List of issues with verbatim quoted evidence, or empty if no issues",
    )
    verdict: str = Field(
        ...,
        description="Explicit verdict: 'NO ISSUES' if acceptable, otherwise 'NEEDS REVISION'",
    )


def evaluate_draft_with_evidence(
    goal: str,
    draft: str,
    llm: Client,
    model: str = "gemini-3.5-flash-lite",
) -> EvidenceCriticResult:
    """Critic prompt requiring quoted evidence for flaws and explicitly permitting 'no issues'."""
    prompt = f"""You are a precise, objective code reviewer evaluating a draft against the specified goal.

GOAL:
{goal}

DRAFT TO REVIEW:
{draft}

CRITIQUE RULES:
1. Grounding via Quoted Evidence: For EVERY issue found, you MUST provide an exact, verbatim quote from the draft in `quoted_evidence`. Do NOT invent or paraphrase snippets.
2. Permit 'No Issues': If the draft correctly satisfies the goal without logic errors or crashes, you MUST return `has_issues=False`, `issues=[]`, and `verdict='NO ISSUES'`. Do NOT invent nitpicks or hallucinate problems on good code.
3. If genuine flaws exist, set `has_issues=True`, provide quoted evidence for each, and set `verdict='NEEDS REVISION'`.
"""
    response = llm.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=EvidenceCriticResult,
        ),
    )
    return getattr(response, "parsed", None) or EvidenceCriticResult.model_validate_json(
        response.text
    )

In [10]:
# Test the critic on 5 seeded flawed drafts and 5 good drafts
goal_task = "Implement a function `def calculate_average(nums: list[float]) -> float | None` to safely return the average of a list of numbers, or None if empty."

flawed_drafts = [
    # 1. ZeroDivisionError on empty list
    "def calculate_average(nums):\n    return sum(nums) / len(nums)",
    # 2. Integer division truncating float values
    "def calculate_average(nums):\n    if not nums: return None\n    return sum(nums) // len(nums)",
    # 3. Off-by-one error in denominator
    "def calculate_average(nums):\n    if not nums: return None\n    return sum(nums) / (len(nums) - 1)",
    # 4. Modifying input list and returning wrong type
    "def calculate_average(nums):\n    if not nums: return None\n    nums.clear()\n    return 'done'",
    # 5. Typo in variable name (NameError)
    "def calculate_average(nums):\n    if not nums: return None\n    total = sum(nums)\n    return totl / len(nums)",
]

good_drafts = [
    # 1. Clean guard clause
    "def calculate_average(nums: list[float]) -> float | None:\n    if not nums:\n        return None\n    return sum(nums) / len(nums)",
    # 2. Explicit length check
    "def calculate_average(nums: list[float]) -> float | None:\n    if len(nums) == 0:\n        return None\n    return float(sum(nums)) / len(nums)",
    # 3. Defensive float check and docstring
    'def calculate_average(nums: list[float]) -> float | None:\n    """Return arithmetic mean or None if list is empty."""\n    if not nums:\n        return None\n    return sum(nums) / float(len(nums))',
    # 4. Iterative accumulator
    "def calculate_average(nums: list[float]) -> float | None:\n    if not nums:\n        return None\n    total = 0.0\n    for n in nums:\n        total += float(n)\n    return total / len(nums)",
    # 5. Using built-in statistics module safely
    "from statistics import mean, StatisticsError\ndef calculate_average(nums: list[float]) -> float | None:\n    try:\n        return float(mean(nums))\n    except StatisticsError:\n        return None",
]

print("=== Evaluating 5 Flawed Drafts (Expect has_issues=True with quoted evidence) ===")
flawed_correct = 0
for idx, draft in enumerate(flawed_drafts, 1):
    result = evaluate_draft_with_evidence(goal_task, draft, google_client)
    has_expected = result.has_issues is True
    flawed_correct += int(has_expected)
    print(f"\nFlawed Draft #{idx}: Verdict: {result.verdict} | Issues: {len(result.issues)}")
    for issue in result.issues:
        print(f"  - Quoted Evidence: {issue.quoted_evidence!r}")
        print(f"    Explanation: {issue.explanation}")

print("\n=== Evaluating 5 Good Drafts (Expect has_issues=False / NO ISSUES) ===")
good_correct = 0
for idx, draft in enumerate(good_drafts, 1):
    result = evaluate_draft_with_evidence(goal_task, draft, google_client)
    has_expected = result.has_issues is False
    good_correct += int(has_expected)
    print(
        f"Good Draft #{idx}: Verdict: {result.verdict} | has_issues={result.has_issues} | Issues: {result.issues}"
    )

print("\nSummary Accuracy:")
print(f"Flawed Detection: {flawed_correct}/5 (True Positives)")
print(f"Good Acceptance:  {good_correct}/5 (True Negatives, no false alarms)")
print(f"Overall Accuracy: {(flawed_correct + good_correct) / 10 * 100:.1f}%")

=== Evaluating 5 Flawed Drafts (Expect has_issues=True with quoted evidence) ===

Flawed Draft #1: Verdict: NEEDS REVISION | Issues: 1
  - Quoted Evidence: 'return sum(nums) / len(nums)'
    Explanation: The function does not handle the case where the input list is empty, which will result in a ZeroDivisionError when dividing by len(nums) (0). The goal requires returning None if the list is empty.

Flawed Draft #2: Verdict: NEEDS REVISION | Issues: 1
  - Quoted Evidence: 'return sum(nums) // len(nums)'
    Explanation: The draft uses integer division (//) instead of standard division (/), which returns an integer instead of a float and truncates the decimal part, violating the expected return type of float.

Flawed Draft #3: Verdict: NEEDS REVISION | Issues: 1
  - Quoted Evidence: 'return sum(nums) / (len(nums) - 1)'
    Explanation: The function calculates the average by dividing the sum by (len(nums) - 1) instead of len(nums), which results in an incorrect mathematical average.

Flaw

### Task 6

In [11]:
class RubricEvaluation(BaseModel):
    """Evaluation result from anchored rubric judge."""

    score: int = Field(..., ge=1, le=5, description="Rubric score from 1 to 5")
    rationale: str = Field(..., description="Concise justification referencing the rubric anchors")


def judge_with_anchored_rubric(
    task_description: str,
    candidate: str,
    llm: Client,
    model: str = "gemini-3.5-flash-lite",
    max_retries: int = 3,
) -> RubricEvaluation:
    """Judge a candidate solution using an anchored 1-5 scale rubric with rate-limit retry."""
    rubric_prompt = (
        "You are an expert evaluator scoring candidate solutions on a strict 1-5 scale.\n\n"
        f"TASK:\n{task_description}\n\n"
        "ANCHORED RUBRIC:\n"
        "Score 1: Broken / Incoherent / Syntax error / Crashes immediately.\n"
        "- Anchor Example:\n"
        "  def is_palindrome(s):\n"
        "      return s.reversed()  # AttributeError\n\n"
        "Score 2: Runs but misses core requirements or severely flawed logic.\n"
        "- Anchor Example:\n"
        "  def is_palindrome(s):\n"
        "      return s == s[::-1]  # Raw comparison without case/punctuation handling\n\n"
        "Score 3: Satisfies core logic partially; misses critical edge cases (e.g. punctuation, spaces).\n"
        "- Anchor Example:\n"
        "  def is_palindrome(s):\n"
        "      s = s.lower()\n"
        "      return s == s[::-1]  # Handles lower, ignores punctuation/spaces\n\n"
        "Score 4: Correct and satisfies all functional requirements, but lacks typing, docstrings, or validation.\n"
        "- Anchor Example:\n"
        "  def is_palindrome(s):\n"
        "      clean = [c.lower() for c in s if c.isalnum()]\n"
        "      return clean == clean[::-1]\n\n"
        "Score 5: Exemplary; robust, includes type hints, docstring, input validation, and optimal handling.\n"
        "- Anchor Example:\n"
        "  def is_palindrome(s: str) -> bool:\n"
        '      """Check if string is palindrome, ignoring case and non-alphanumeric chars."""\n'
        "      if not isinstance(s, str):\n"
        "          return False\n"
        "      filtered = [c.lower() for c in s if c.isalnum()]\n"
        "      return filtered == filtered[::-1]\n\n"
        f"CANDIDATE TO EVALUATE:\n{candidate}\n\n"
        "Assign a score from 1 to 5 based strictly on matching the candidate to the anchored criteria. Provide a concise rationale."
    )
    for attempt in range(max_retries):
        try:
            response = llm.models.generate_content(
                model=model,
                contents=rubric_prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=RubricEvaluation,
                ),
            )
            return getattr(response, "parsed", None) or RubricEvaluation.model_validate_json(
                response.text
            )
        except Exception as e:
            if ("429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)) and attempt < max_retries - 1:
                time.sleep(20 * (attempt + 1))
            else:
                raise
    raise RuntimeError("Failed after max retries due to rate limits.")

In [12]:
# Task definition and 20 candidate outputs with ground-truth human labels (1 to 5)
task_desc = (
    "Implement `is_palindrome(s: str) -> bool` to check if a string is a palindrome, "
    "ignoring case and non-alphanumeric characters, with proper typing and docstrings."
)

dataset = [
    # Score 1 Candidates (Broken, invalid, wrong logic)
    ("def is_palindrome(s):\n    return len(s) > 0", 1),
    ("def is_palindrome(s):\n    print(s == s[::-1])", 1),
    ("def is_palindrome(s):\n    return s.reverse()", 1),
    (
        "def is_palindrome(s):\n    for i in range(len(s)):\n        if s[i] == s[i+1]: return True",
        1,
    ),
    # Score 2 Candidates (Direct comparison without case/punctuation normalization)
    ("def is_palindrome(s):\n    return s == s[::-1]", 2),
    ("def is_palindrome(s):\n    return list(s) == list(reversed(s))", 2),
    (
        "def is_palindrome(s):\n    rev = ''\n    for c in s:\n        rev = c + rev\n    return rev == s",
        2,
    ),
    (
        "def is_palindrome(s):\n    n = len(s)\n    for i in range(n // 2):\n        if s[i] != s[n - 1 - i]: return False\n    return True",
        2,
    ),
    # Score 3 Candidates (Handles case or spaces, but misses non-alphanumeric punctuation)
    ("def is_palindrome(s):\n    s = s.lower()\n    return s == s[::-1]", 3),
    ("def is_palindrome(s):\n    s = s.replace(' ', '').lower()\n    return s == s[::-1]", 3),
    ("def is_palindrome(s):\n    s = s.strip().lower()\n    return s == s[::-1]", 3),
    (
        "def is_palindrome(s):\n    t = s.lower()\n    return [c for c in t] == [c for c in reversed(t)]",
        3,
    ),
    # Score 4 Candidates (Fully functional and correct, but missing type hints or docstrings)
    (
        "def is_palindrome(s):\n    clean = [c.lower() for c in s if c.isalnum()]\n    return clean == clean[::-1]",
        4,
    ),
    (
        "def is_palindrome(s):\n    import re\n    cleaned = re.sub(r'[^a-zA-Z0-9]', '', s).lower()\n    return cleaned == cleaned[::-1]",
        4,
    ),
    (
        "def is_palindrome(s):\n    chars = [c.lower() for c in s if c.isalnum()]\n    i, j = 0, len(chars) - 1\n    while i < j:\n        if chars[i] != chars[j]: return False\n        i += 1\n        j -= 1\n    return True",
        4,
    ),
    (
        "def is_palindrome(s: str) -> bool:\n    c = ''.join(ch.lower() for ch in s if ch.isalnum())\n    return c == c[::-1]",
        4,
    ),
    # Score 5 Candidates (Exemplary: type hints, docstring, validation, robust handling)
    (
        'def is_palindrome(s: str) -> bool:\n    """Check whether string is palindrome ignoring non-alphanumeric chars and case."""\n    if not isinstance(s, str):\n        return False\n    filtered = [ch.lower() for ch in s if ch.isalnum()]\n    return filtered == filtered[::-1]',
        5,
    ),
    (
        'def is_palindrome(s: str) -> bool:\n    """Check palindrome property using two-pointer approach."""\n    if not isinstance(s, str):\n        return False\n    i, j = 0, len(s) - 1\n    while i < j:\n        while i < j and not s[i].isalnum(): i += 1\n        while i < j and not s[j].isalnum(): j -= 1\n        if s[i].lower() != s[j].lower(): return False\n        i += 1\n        j -= 1\n    return True',
        5,
    ),
    (
        'import re\n\ndef is_palindrome(s: str) -> bool:\n    """Determine if string is a palindrome with regex normalization."""\n    if not isinstance(s, str):\n        return False\n    cleaned = re.sub(r\'[^a-zA-Z0-9]\', \'\', s).lower()\n    return cleaned == cleaned[::-1]',
        5,
    ),
    (
        'def is_palindrome(s: str) -> bool:\n    """Verify palindrome property case-insensitively and ignoring punctuation."""\n    if not s:\n        return True\n    chars = [char.casefold() for char in s if char.isalnum()]\n    return chars == chars[::-1]',
        5,
    ),
]

human_scores = []
judge_scores = []

print(f"Evaluating {len(dataset)} outputs with the Anchored Rubric Judge...\n")
print(f"{'#':<3} | {'Human':<5} | {'Judge':<5} | {'Diff':<4} | {'Rationale'}")
print("-" * 80)

for idx, (code, human_label) in enumerate(dataset, 1):
    eval_result = judge_with_anchored_rubric(task_desc, code, google_client)
    human_scores.append(human_label)
    judge_scores.append(eval_result.score)
    diff = abs(human_label - eval_result.score)
    print(
        f"{idx:<3} | {human_label:<5} | {eval_result.score:<5} | {diff:<4} | {eval_result.rationale[:55]}..."
    )
    if idx < len(dataset):
        time.sleep(4.5)  # Pace requests to comply with 15 RPM free-tier quota

# Compute agreement metrics
total = len(human_scores)
exact_match = sum(1 for h, j in zip(human_scores, judge_scores) if h == j)
adjacent_match = sum(1 for h, j in zip(human_scores, judge_scores) if abs(h - j) <= 1)
mae = sum(abs(h - j) for h, j in zip(human_scores, judge_scores)) / total

print("\n" + "=" * 50)
print("=== AGREEMENT METRICS ON 20 OUTPUTS ===")
print(f"Exact Agreement:    {exact_match}/{total} ({exact_match / total * 100:.1f}%)")
print(
    f"Adjacent Agreement: {adjacent_match}/{total} ({adjacent_match / total * 100:.1f}%) [within ±1]"
)
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print("=" * 50)

Evaluating 20 outputs with the Anchored Rubric Judge...

#   | Human | Judge | Diff | Rationale
--------------------------------------------------------------------------------
1   | 1     | 1     | 0    | The candidate solution does not implement a palindrome ...
2   | 1     | 1     | 0    | The candidate solution prints a raw string comparison i...
3   | 1     | 1     | 0    | The candidate solution attempts to call a non-existent ...
4   | 1     | 1     | 0    | The candidate solution has broken logic and will immedi...
5   | 2     | 2     | 0    | The candidate implementation uses a raw string reversal...
6   | 2     | 2     | 0    | The candidate's code runs a raw list comparison without...
7   | 2     | 2     | 0    | The candidate implementation runs, but it only performs...
8   | 2     | 2     | 0    | The candidate implementation runs, but it performs a ra...
9   | 3     | 3     | 0    | The candidate implementation handles case conversion by...
10  | 3     | 3     | 0    | Th

### Task 7

implemented in test_position_bias.py

### Task 8

In [ ]:
class ReflectionLesson(BaseModel):
    """Post-run reflection lesson model."""

    lesson: str = Field(..., description="Short, concise verified lesson learned from failure")


async def generate_code(prompt: str, client: Client, lessons: list[str] | None = None) -> str:
    """Generate Python code, optionally conditioned on past episodic lessons."""
    sys_instruction = (
        "You are a Python expert. Provide ONLY executable Python code, no markdown or explanation."
    )
    if lessons:
        sys_instruction += "\n\nPAST VERIFIED LESSONS TO FOLLOW:\n" + "\n".join(
            f"- {lesson}" for lesson in lessons
        )

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=sys_instruction,
            temperature=0.0,
        ),
    )
    code = response.text.strip()
    if code.startswith("```"):
        code = re.sub(r"^```(?:python)?\n?", "", code)
        code = re.sub(r"\n?```$", "", code)
    return code.strip()


def verify_code(code: str) -> tuple[bool, str]:
    """Deterministic verifier testing base and plus-addressed email username extraction."""
    scope: dict[str, Any] = {}
    try:
        exec(code, scope)  # noqa: S102
        func = scope.get("extract_username")
        if not func:
            return False, "Function `extract_username` not defined."
        if func("alice@example.com") != "alice":
            return False, "Failed basic extraction for 'alice@example.com'"
        res = func("john.doe+newsletter@example.com")
        if res != "john.doe":
            return False, f"Failed for plus-addressing: got '{res}', expected 'john.doe'"
        return True, "All tests passed!"
    except Exception as e:  # noqa: BLE001
        return False, f"Execution error: {e}"


async def reflect_and_learn(task: str, code: str, error: str, client: Client) -> str:
    """Analyze failure post-run and extract a concise verified lesson."""
    prompt = (
        f"TASK: {task}\n"
        f"FAILED CODE:\n{code}\n"
        f"TEST FAILURE ERROR:\n{error}\n\n"
        "Analyze why the code failed and formulate a single concise, actionable lesson (1-2 sentences) "
        "that future code generation runs must follow to avoid this exact mistake."
    )
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ReflectionLesson,
        ),
    )
    parsed = getattr(response, "parsed", None) or ReflectionLesson.model_validate_json(
        response.text
    )
    return parsed.lesson

In [16]:
# Initialize genai-core episodic memory store
store = InMemoryMemoryStore()
episodic_memory = EpisodicMemory(store=store)
user_id = "agent_developer"
task_prompt = "Write a Python function `extract_username(email: str) -> str` to extract the base username from an email."

print("=" * 65)
print("--- RUN 1: Zero-shot Generation (Without Episodic Lessons) ---")
run1_code = await generate_code(task_prompt, google_client)
print(f"Generated Code (Run 1):\n{run1_code}\n")

passed_1, err_1 = verify_code(run1_code)
print(f"Run 1 Verification: {'PASSED' if passed_1 else 'FAILED'}")
print(f"Error / Feedback: {err_1}\n")

if not passed_1:
    print("--- POST-RUN REFLECTION ---")
    lesson = await reflect_and_learn(task_prompt, run1_code, err_1, google_client)
    print(f"Extracted Verified Lesson: {lesson}\n")

    # Record verified lesson into genai-core episodic memory
    episode = await episodic_memory.record(
        user_id=user_id,
        action_type="post_run_reflection",
        parameters={"task": "extract_username"},
        result_summary=lesson,
        metadata={"verified": True, "failed_input": "john.doe+newsletter@example.com"},
    )
    print(f"Saved into genai-core Episodic Memory! Episode ID: {episode.id}\n")

# Retrieve past lessons from episodic memory
past_episodes = await episodic_memory.get_episodes(
    user_id=user_id, action_type="post_run_reflection"
)
retrieved_lessons = [ep.result_summary for ep in past_episodes if ep.result_summary]

print("=" * 65)
print(f"--- RUN 2: Generation Conditioned on Episodic Memory ({len(retrieved_lessons)} lesson) ---")
run2_code = await generate_code(task_prompt, google_client, lessons=retrieved_lessons)
print(f"Generated Code (Run 2):\n{run2_code}\n")

passed_2, err_2 = verify_code(run2_code)
print(f"Run 2 Verification: {'PASSED' if passed_2 else 'FAILED'}")
print(f"Status: {err_2}")
print("=" * 65)

--- RUN 1: Zero-shot Generation (Without Episodic Lessons) ---
Generated Code (Run 1):
def extract_username(email: str) -> str:
    return email.split('@')[0]

Run 1 Verification: FAILED
Error / Feedback: Failed for plus-addressing: got 'john.doe+newsletter', expected 'john.doe'

--- POST-RUN REFLECTION ---
Extracted Verified Lesson: When extracting a username from an email address, handle plus-addressing by removing any tags introduced by the '+' character before the domain split.

Saved into genai-core Episodic Memory! Episode ID: 9aa58829-2ba9-4c58-8cc2-017b1ca9d71b

--- RUN 2: Generation Conditioned on Episodic Memory (1 lesson) ---
Generated Code (Run 2):
def extract_username(email: str) -> str:
    local_part, domain = email.split('@')
    if '+' in local_part:
        local_part = local_part.split('+')[0]
    return local_part

Run 2 Verification: PASSED
Status: All tests passed!
